# CD CosMx resections: merge per-FOV LIANA results

Collects the per-FOV LIANA outputs of `06_resection_liana.ipynb` (`<run>/Processed/liana/fov_<n>.h5ad`) for all resection runs into one per-cell table:

1. per-cell LR scores (`X`) of every FOV, concatenated per run (`<run>/Processed/<run>_liana_lrspot_X_combined.parquet`), then over all runs;
2. cell IDs prefixed with patient and section (`patient_sec`), missing scores set to 0 (no interaction);
3. cell annotation from `combined/all_samples.parquet`; AUCell Tregs are kept only if they are strict Tregs, i.e. listed in
   `combined/strict_treg_pt_transfer_labels.csv` (one row per strict Treg of the 6,554-cell resection query; only its cell IDs are used here);
4. output `combined/spot_lr_df.parquet` (cells x LR pairs), read by `11_neighbourhoods_local_tables.py` and `13`-`15`.

In [ ]:
import pandas as pd
import os
import anndata as ad
import gc

In [ ]:
sample_list =[
#'Cos1_6KDC_strict18888_A5_0715_11_08_2025_14_21_12_760',                
'Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807',          
'Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437',       
'Cos1_6KDC_strictCD_R1_B4_0715_11_08_2025_14_20_23_133',     
'Cos2_6KDC_strict18888_A10_0715_Alan_data_test_11_08_2025_14_25_11_455',  
'Cos2_6KDC_strict18888_A6_0715_Alan_Test_11_08_2025_14_23_09_489',  
'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615',                  
'Cos3_6KDC_strictCD_R1_B5_0716_11_08_2025_14_17_56_277',                
'Cos4_6KDC_strict11808_A4_0716_14_10_2025_12_34_38_727',
'Cos4_6KDC_strictCD_R1_B7_0716_11_08_2025_14_18_41_738',
'Cos5_6KDC_strict43563_SB_A11_091725BP7_09_10_2025_13_42_42_775',         
'Cos5_6KDC_strict43563_SB_A4_091725BP8_09_10_2025_13_43_50_952'  
]

In [ ]:
comb_lst = {}
for i in sample_list: 
    if 'SB' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[2]
        comb = patient+'_'+section
    elif '11808' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        machine = i.split('_')[0]
        comb = patient+'_'+section+'_'+machine
    else:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        comb = patient+'_'+section
    comb_lst[i]=comb

## Per-cell LR scores of all runs

In [ ]:
def concatenate_fov_X_per_sample(base_dir, sample_folders, output_suffix='_liana_lrspot_X_combined.parquet'):
    """Per run: concatenate the LR score matrix X (cells x LR pairs) of every FOV h5ad, add 'fov' and 'sample', save as parquet."""
    for sample in sample_folders:
        liana_path = os.path.join(base_dir, sample, "Processed", "liana")
        output_path = os.path.join(base_dir, sample, "Processed", f"{sample}{output_suffix}")
        files = [f for f in os.listdir(liana_path) if f.endswith('.h5ad')]
        try:
            df_list = []
            for file in files:
                adata = ad.read_h5ad(os.path.join(liana_path, file))
                X_df = pd.DataFrame(adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
                X_df['fov'] = os.path.splitext(file)[0]
                X_df['sample'] = sample
                df_list.append(X_df)
                del adata, X_df
                gc.collect()
            sample_combined = pd.concat(df_list, axis=0, ignore_index=False)
            sample_combined.index.name = "obs_id"
            sample_combined.to_parquet(output_path)
            del df_list, sample_combined
            gc.collect()
        except Exception as e:
            print(f"Error processing {sample}: {e}")
            gc.collect()


def concatenate_all_samples_X(base_dir, sample_folders, output_suffix='_liana_lrspot_X_combined.parquet'):
    """Concatenate the per-run parquet files; index = (sample, obs_id)."""
    all_samples_list = []
    for i, sample in enumerate(sample_folders, 1):
        all_samples_list.append(pd.read_parquet(os.path.join(base_dir, sample, "Processed", f"{sample}{output_suffix}")))
        if i % 10 == 0:
            gc.collect()
    all_combined = pd.concat(all_samples_list, axis=0, ignore_index=False)
    if all_combined.index.name is None:
        all_combined.index.name = "obs_id"
    obs_index_name = all_combined.index.name
    all_combined = all_combined.reset_index()
    all_combined.set_index(['sample', obs_index_name], inplace=True)
    del all_samples_list
    gc.collect()
    return all_combined

In [ ]:
base_dir = '/path/to/cosmx_data/cd_resection'
sample_folders = sample_list
concatenate_fov_X_per_sample(base_dir, sample_folders)

In [ ]:
lr_x_df = concatenate_all_samples_X(base_dir,sample_folders)

In [ ]:
spot_lr_df_filled = lr_x_df.fillna(0.0)

In [ ]:
spot_lr_df_filled['patient_sec'] = spot_lr_df_filled.index.get_level_values('sample').map(comb_lst)

In [ ]:
spot_lr_df_filled['cell'] = spot_lr_df_filled.index.get_level_values('obs_id')

In [ ]:
spot_lr_df_filled['new_idx']=[i+'_'+j for i,j in zip(spot_lr_df_filled['patient_sec'].tolist(), spot_lr_df_filled['cell'].tolist())]


In [ ]:
spot_lr_df_filled.index = spot_lr_df_filled['new_idx'].tolist()

In [ ]:
spot_lr_df_filled=spot_lr_df_filled.drop(['patient_sec','cell','new_idx','fov'],axis = 1)

In [ ]:
spot_lr_df=spot_lr_df_filled

In [ ]:
spot_anno =pd.read_parquet('/path/to/cosmx_data/cd_resection/combined/all_samples.parquet')

In [ ]:
spot_anno=spot_anno[['fov','source_file', 'cell_type', 'cell_type_short', 'cell_category',
       'cell_type_general', 'patient_sec', 'patient']]

In [ ]:
spot_anno['new_idx']=[i+'_'+j for i,j in zip(spot_anno['patient_sec'].tolist(), spot_anno.index.tolist())]

spot_anno.index =spot_anno['new_idx'].tolist()
spot_anno=spot_anno.drop('new_idx',axis = 1)

In [ ]:
spot_anno=spot_anno[spot_anno.index.isin(spot_lr_df.index.tolist())]

In [ ]:
spot_anno = spot_anno.reindex(spot_lr_df.index)

In [ ]:
tregs = pd.read_csv("/path/to/cosmx_data/cd_resection/combined/strict_treg_pt_transfer_labels.csv")

In [ ]:
# strict-Treg table: its cell IDs decide which AUCell Tregs are kept below
tregs['early_late']=['Late Treg' if i>=3 else 'Early Treg' for i in tregs['predicted.id'].tolist()]
tregs.index = tregs['Unnamed: 0'].tolist()

In [ ]:
spot_anno = spot_anno.merge(tregs[['early_late']],how='left',left_index=True, right_index=True)

In [ ]:
# cell-type grouping; AUCell Tregs without a strict-Treg row are marked 'drop'
network_anno =[]
for i,j,k,m in zip (spot_anno['cell_category'].tolist(),spot_anno['cell_type_general'].tolist(),spot_anno['cell_type_short'].tolist(),spot_anno['early_late'].tolist()):
    if i == 'Myeloid':
        if 'DC' in j:
            network_anno.append(j)
        elif k in ['Mac S+XS-','Mac S+SG+']:
            network_anno.append('Mac S')
        elif k == 'Mac S+M+P+':
            network_anno.append('Mac S_M_P')
        elif k == 'Mac S+M+S+':
            network_anno.append('Mac S_M_S')
        elif 'CD9' in j:
            network_anno.append('CD9 Mac')
        else:
            network_anno.append(k)
    elif i == 'Connective':
        if 'OGN' in j:
            network_anno.append('OGN_RSPO3 Fib')
        elif 'ADAMDEC1' in j:
            network_anno.append('ADAMDEC1 Fib')
        elif 'PLA2G2A' in j:
            network_anno.append('PLA2G2A Fib')
        elif 'SELENOP' in j:
            network_anno.append('SELENOP Fib')
        else: 
            network_anno.append(j)
    elif i == 'T':
        if j == 'Treg':
            if pd.notna(m):
                network_anno.append(m)
            else:
                network_anno.append('drop')
        elif 'Tfh' in j:
            network_anno.append('Tfh')
        else:
            network_anno.append(j)
    else:
        network_anno.append(i)

In [ ]:
spot_anno['network_anno']=network_anno

In [ ]:
spot_anno = spot_anno[spot_anno['network_anno']!='drop']

In [ ]:
spot_lr_df=spot_lr_df[spot_lr_df.index.isin(spot_anno.index.tolist())]

In [ ]:
spot_lr_df.to_parquet('/path/to/cosmx_data/cd_resection/combined/spot_lr_df.parquet')